In [ ]:
from kaggle_environments import make


EPISODE_STEPS = 720
TURNS_PER_DAY = 24
DAYS = 30

BOARD_SIZE = 10
SHED_CAPACITY = 100

print("=" * 80)
print("KAGGRICULTURE COMPETITION AGENT")
print("=" * 80)

print("Days:", DAYS)
print("Turns:", EPISODE_STEPS)
print("Board:", BOARD_SIZE, "x", BOARD_SIZE)

In [ ]:
CROPS = {
    "WHEAT": {
        "seed": 10,
        "first_yield_day": 2,
        "max_yield_day": 4,
        "max_yield": 6,
        "ongoing": False
    },

    "CARROT": {
        "seed": 20,
        "first_yield_day": 2,
        "max_yield_day": 3,
        "max_yield": 4,
        "ongoing": False
    },

    "TOMATO": {
        "seed": 50,
        "first_yield_day": 8,
        "max_yield_day": 8,
        "max_yield": 4,
        "ongoing": True
    },

    "STRAWBERRY": {
        "seed": 100,
        "first_yield_day": 10,
        "max_yield_day": 10,
        "max_yield": 4,
        "ongoing": True
    },

    "MELON": {
        "seed": 80,
        "first_yield_day": 10,
        "max_yield_day": 12,
        "max_yield": 6,
        "ongoing": False
    }
}


LAND_ORDER = [
    "NE",
    "SW",
    "SE"
]

LAND_PRICES = {
    "NE": 1000,
    "SW": 2000,
    "SE": 4000
}

FARM_HAND_COST_MULT = 10

FIB = [
    1, 1, 2, 3, 5, 8, 13,
    21, 34, 55, 89, 144
]


print("Crops:", CROPS)
print("Land:", LAND_PRICES)
print("Hand multiplier:", FARM_HAND_COST_MULT)

In [ ]:
QUADRANT_TILES = {

    "NW": [
        (x, y)
        for y in range(0, 5)
        for x in range(0, 5)
    ],

    "NE": [
        (x, y)
        for y in range(0, 5)
        for x in range(5, 10)
    ],

    "SW": [
        (x, y)
        for y in range(5, 10)
        for x in range(0, 5)
    ],

    "SE": [
        (x, y)
        for y in range(5, 10)
        for x in range(5, 10)
    ]
}


def unlocked_tiles(farm):

    unlocked = farm.get(
        "unlocked_quadrants",
        ["NW"]
    )

    tiles = []

    for q in ["NW", "NE", "SW", "SE"]:

        if q in unlocked:

            tiles.extend(
                QUADRANT_TILES[q]
            )

    return tiles


print(
    "NW tiles:",
    len(QUADRANT_TILES["NW"])
)

print(
    "Full farm:",
    sum(
        len(v)
        for v in QUADRANT_TILES.values()
    )
)

In [ ]:
def get_player_farm(obs):

    player = obs.get(
        "player",
        0
    )

    farms = obs.get(
        "farms",
        []
    )

    if not farms:
        return None

    if player >= len(farms):
        return None

    return farms[player]


def get_private(obs):

    private = obs.get(
        "private",
        {}
    )

    if not isinstance(
        private,
        dict
    ):
        return {}

    return private


def get_shed(obs):

    private = get_private(obs)

    shed = private.get(
        "shed",
        {}
    )

    if not isinstance(
        shed,
        dict
    ):
        return {}

    return shed


def get_seeds(obs):

    private = get_private(obs)

    seeds = private.get(
        "seeds",
        {}
    )

    if not isinstance(
        seeds,
        dict
    ):
        return {}

    return seeds


def get_market_prices(obs):

    market = obs.get(
        "market",
        {}
    )

    if not isinstance(
        market,
        dict
    ):
        return {}

    prices = market.get(
        "prices",
        {}
    )

    if not isinstance(
        prices,
        dict
    ):
        return {}

    return prices


def get_day(obs):

    return int(
        obs.get(
            "day",
            0
        )
    )


def tile_at(farm, x, y):

    try:
        return farm[
            "tiles"
        ][y][x]

    except Exception:
        return None

In [ ]:
def normalize_position(value):

    if (
        isinstance(value, (list, tuple))
        and
        len(value) >= 2
    ):

        try:

            return (
                int(value[0]),
                int(value[1])
            )

        except Exception:
            pass

    return None


def extract_hand_position(hand):

    if isinstance(
        hand,
        dict
    ):

        for key in [
            "position",
            "pos",
            "farmer"
        ]:

            if key in hand:

                p = normalize_position(
                    hand[key]
                )

                if p is not None:
                    return p

    return normalize_position(
        hand
    )


def get_unit_positions(farm):

    positions = []

    farmer_pos = normalize_position(
        farm.get(
            "farmer",
            [4, 4]
        )
    )

    if farmer_pos is None:
        farmer_pos = (4, 4)

    positions.append(
        farmer_pos
    )

    hands = farm.get(
        "hands",
        []
    )

    if not isinstance(
        hands,
        list
    ):
        hands = []

    for hand in hands:

        p = extract_hand_position(
            hand
        )

        if p is None:
            p = farmer_pos

        positions.append(
            p
        )

    return positions

In [ ]:
def is_plant(tile):

    return (
        isinstance(tile, dict)
        and
        tile.get("kind") == "PLANT"
    )


def is_weed(tile):

    if not isinstance(
        tile,
        dict
    ):
        return False

    return tile.get(
        "kind"
    ) in {
        "WEED",
        "weed"
    }


def is_empty(tile):

    return tile is None


def is_mature(tile, day):

    if not is_plant(tile):
        return False

    crop = tile.get(
        "crop"
    )

    if crop not in CROPS:
        return False

    planted_day = tile.get(
        "planted_day",
        day
    )

    age = (
        day
        -
        int(planted_day)
    )

    return (
        age >=
        CROPS[crop][
            "max_yield_day"
        ]
    )


def needs_water(tile):

    return (
        is_plant(tile)
        and
        not tile.get(
            "watered_today",
            False
        )
    )

In [ ]:
def build_tasks(obs):

    farm = get_player_farm(
        obs
    )

    if farm is None:
        return []

    day = get_day(
        obs
    )

    seeds = get_seeds(
        obs
    )

    tasks = []

    for x, y in unlocked_tiles(
        farm
    ):

        tile = tile_at(
            farm,
            x,
            y
        )


        if is_mature(
            tile,
            day
        ):

            tasks.append(
                (
                    0,
                    (x, y),
                    "HARVEST"
                )
            )

            continue


        if needs_water(
            tile
        ):

            tasks.append(
                (
                    1,
                    (x, y),
                    "WATER"
                )
            )

            continue

        if (
            is_empty(tile)
            and
            seeds.get(
                "MELON",
                0
            ) > 0
        ):

            tasks.append(
                (
                    2,
                    (x, y),
                    "PLANT"
                )
            )

            continue


        if is_weed(tile):

            tasks.append(
                (
                    3,
                    (x, y),
                    "DIG"
                )
            )

    return tasks

In [ ]:
def move_toward(
    current,
    target
):

    cx, cy = current
    tx, ty = target

    dx = tx - cx
    dy = ty - cy

    if dx == 0 and dy == 0:
        return None

    if abs(dx) >= abs(dy):

        if dx > 0:
            return ["EAST"]

        if dx < 0:
            return ["WEST"]

    else:

        if dy > 0:
            return ["SOUTH"]

        if dy < 0:
            return ["NORTH"]

    return ["PASS"]


def distance(
    a,
    b
):

    return (
        abs(a[0] - b[0])
        +
        abs(a[1] - b[1])
    )

In [ ]:
def unit_actions(obs):

    farm = get_player_farm(
        obs
    )

    if farm is None:
        return (
            ["PASS"],
            []
        )

    day = get_day(
        obs
    )

    seeds = get_seeds(
        obs
    )

    shed = get_shed(
        obs
    )

    positions = get_unit_positions(
        farm
    )

    unit_count = len(
        positions
    )

    if unit_count <= 0:

        return (
            ["PASS"],
            []
        )


    melon_in_shed = int(
        shed.get(
            "MELON",
            0
        )
    )

    harvest_capacity = max(
        0,
        (
            SHED_CAPACITY
            -
            melon_in_shed
        )
        // 6
    )

    harvest_capacity = min(
        harvest_capacity,
        15
    )


    plant_budget = int(
        seeds.get(
            "MELON",
            0
        )
    )

    tasks = build_tasks(
        obs
    )

    reserved = set()

    actions = []

    harvest_used = 0
    plant_used = 0


    for unit_id, current in enumerate(
        positions
    ):

        best = None
        best_key = None

        for priority, target, operation in tasks:

            if target in reserved:
                continue

            if (
                operation == "HARVEST"
                and
                harvest_used >= harvest_capacity
            ):
                continue

            if (
                operation == "PLANT"
                and
                plant_used >= plant_budget
            ):
                continue

            d = distance(
                current,
                target
            )

            tie = (
                target[0]
                +
                target[1] * BOARD_SIZE
                +
                unit_id
            ) % max(
                1,
                unit_count
            )

            key = (
                priority,
                d,
                tie
            )

            if (
                best_key is None
                or
                key < best_key
            ):

                best_key = key
                best = (
                    target,
                    operation
                )

        if best is None:

            actions.append(
                ["PASS"]
            )

            continue

        target, operation = best

        reserved.add(
            target
        )


        if current == target:

            if operation == "PLANT":

                actions.append(
                    [
                        "PLANT",
                        "MELON"
                    ]
                )

                plant_used += 1

            else:

                actions.append(
                    [
                        operation
                    ]
                )

                if operation == "HARVEST":
                    harvest_used += 1

            continue


        movement = move_toward(
            current,
            target
        )

        if movement is None:
            movement = ["PASS"]

        actions.append(
            movement
        )


    farmer_action = (
        actions[0]
        if actions
        else
        ["PASS"]
    )

    hand_actions = actions[1:]

    return (
        farmer_action,
        hand_actions
    )

In [ ]:
def hire_cost(
    current_hands
):

    idx = int(
        current_hands
    )

    if idx >= len(FIB):

        a = FIB[-2]
        b = FIB[-1]

        while len(FIB) <= idx:

            a, b = b, a + b

            FIB.append(b)

    return (
        FIB[idx]
        *
        FARM_HAND_COST_MULT
    )


def economy_orders(obs):

    farm = get_player_farm(
        obs
    )

    if farm is None:
        return []

    day = get_day(
        obs
    )

    money = float(
        farm.get(
            "money",
            0
        )
    )

    hands = farm.get(
        "hands",
        []
    )

    if not isinstance(
        hands,
        list
    ):
        hands = []

    hand_count = len(
        hands
    )

    unlocked = farm.get(
        "unlocked_quadrants",
        ["NW"]
    )

    if not isinstance(
        unlocked,
        list
    ):
        unlocked = ["NW"]

    seeds = get_seeds(
        obs
    )

    shed = get_shed(
        obs
    )

    prices = get_market_prices(
        obs
    )

    orders = []


    melon_qty = int(
        shed.get(
            "MELON",
            0
        )
    )

    if melon_qty > 0:

        price = float(
            prices.get(
                "MELON",
                250
            )
        )

        if day >= 29:

            sell_qty = melon_qty

        elif price >= 220:

            sell_qty = min(
                melon_qty,
                20
            )

        elif price >= 180:

            sell_qty = min(
                melon_qty,
                15
            )

        elif price >= 140:

            sell_qty = min(
                melon_qty,
                10
            )

        else:

            sell_qty = min(
                melon_qty,
                5
            )

        orders.append(
            [
                "SELL",
                "MELON",
                sell_qty
            ]
        )

        return orders


    if day < 12:

        target_hands = 5

    elif day < 24:

        target_hands = 8

    else:

        target_hands = 8


    if day >= 12:

        target_land = [
            "NE",
            "SW",
            "SE"
        ]

    else:

        target_land = []


    if day < 12:

        target_seeds = 16

    elif day < 25:

        target_seeds = 100

    else:

        target_seeds = 0


    budget = money


    for quadrant in LAND_ORDER:

        if quadrant not in target_land:
            continue

        if quadrant in unlocked:
            continue

        cost = LAND_PRICES[
            quadrant
        ]

        if budget >= cost:

            orders.append(
                [
                    "BUY_LAND"
                ]
            )

            budget -= cost

        else:

            break

        if len(orders) >= 10:
            return orders[:10]


    while (
        hand_count < target_hands
        and
        len(orders) < 10
    ):

        cost = hire_cost(
            hand_count
        )

        if budget < cost:
            break

        orders.append(
            [
                "HIRE"
            ]
        )

        budget -= cost

        hand_count += 1


    current_melon_seeds = int(
        seeds.get(
            "MELON",
            0
        )
    )

    needed = max(
        0,
        target_seeds
        -
        current_melon_seeds
    )

    if needed > 0:

        seed_cost = (
            CROPS["MELON"]["seed"]
        )

        affordable = int(
            budget // seed_cost
        )

        buy_qty = min(
            needed,
            affordable
        )

        if buy_qty > 0:

            orders.append(
                [
                    "BUY_SEED",
                    "MELON",
                    buy_qty
                ]
            )

    return orders[:10]

In [ ]:
def agent(obs):

    try:

        farmer_action, hand_actions = (
            unit_actions(obs)
        )

        market_orders = economy_orders(
            obs
        )

        return {
            "farmer": farmer_action,
            "hands": hand_actions,
            "market": market_orders
        }

    except Exception:

        farm = get_player_farm(
            obs
        )

        hand_count = 0

        if farm is not None:

            hands = farm.get(
                "hands",
                []
            )

            if isinstance(
                hands,
                list
            ):
                hand_count = len(
                    hands
                )

        return {
            "farmer": ["PASS"],
            "hands": [
                ["PASS"]
                for _ in range(
                    hand_count
                )
            ],
            "market": []
        }


print("=" * 80)
print("FINAL AGENT LOADED")
print("=" * 80)

assert callable(agent)

print("agent:", agent)
print("callable")

In [ ]:
test_env = make(
    "kaggriculture",
    configuration={
        "episodeSteps": 720,
        "seed": 0
    },
    debug=True
)

state = test_env.reset()

obs = state[0].observation

action = agent(
    obs
)

print("=" * 80)
print("FIRST ACTION")
print("=" * 80)

print(action)

assert isinstance(
    action,
    dict
)

assert "farmer" in action
assert "hands" in action
assert "market" in action

assert isinstance(
    action["farmer"],
    list
)

assert isinstance(
    action["hands"],
    list
)

assert isinstance(
    action["market"],
    list
)

print()
print("Structure valid")
print("No starter_agent dependency")
print("No external CROPS dependency")

In [ ]:
env = make(
    "kaggriculture",
    configuration={
        "episodeSteps": 720,
        "seed": 0
    },
    debug=True
)

print("=" * 80)
print("RUNNING 720 TURNS")
print("=" * 80)

env.run([
    agent,
    "random"
])

print()
print("Episode completed")
print("Steps:", len(env.steps))

In [ ]:
final = env.steps[-1]

player0 = final[0]

score = float(
    player0.reward
)

farm = player0.observation.farms[0]
private = player0.observation.private

print("=" * 80)
print("FINAL SCORE")
print("=" * 80)

print("Score:", score)
print("Gap:", TARGET_SCORE - score)

print()
print("Status:", player0.status)

print()
print("Day:", player0.observation.day)
print("Hour:", player0.observation.hour)

print()
print("Money:", farm["money"])

print()
print("Land:")
print(
    farm["unlocked_quadrants"]
)

print()
print("Hands:")
print(
    len(
        farm.get(
            "hands",
            []
        )
    )
)

print()
print("Shed:")
print(
    private.get(
        "shed",
        {}
    )
)

print()
print("Seeds:")
print(
    private.get(
        "seeds",
        {}
    )
)

print()
print("=" * 80)

if score >= TARGET_SCORE:

    print("TARGET REACHED")

else:

    print(
        "TARGET NOT REACHED"
    )

print("=" * 80)

In [ ]:
def run_game(seed):

    e = make(
        "kaggriculture",
        configuration={
            "episodeSteps": 720,
            "seed": seed
        },
        debug=True
    )

    e.run([
        agent,
        "random"
    ])

    final_state = e.steps[-1]

    return float(
        final_state[0].reward
    )


results = []

print("=" * 80)
print("MULTI-SEED TEST")
print("=" * 80)

for seed in range(5):

    print(
        "Running seed",
        seed,
        "..."
    )

    s = run_game(
        seed
    )

    results.append(
        s
    )

    print(
        "Score:",
        s
    )

    print()

print("=" * 80)
print("SUMMARY")
print("=" * 80)

for i, s in enumerate(
    results
):

    print(
        f"Seed {i}: {s}"
    )

print()
print("Best:", max(results))
print(
    "Average:",
    sum(results) / len(results)
)

In [ ]:
from pathlib import Path

print("Notebook source:")
print(Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb").exists())

print("Current files:")
for p in Path("/kaggle/working").iterdir():
    print(" -", p.name)

In [ ]:
from pathlib import Path
import inspect
import ast
import types

OUTPUT = Path("/kaggle/working/main.py")

print("=" * 80)
print("BUILDING KAGGRICULTURE main.py")
print("=" * 80)


assert "agent" in globals(), (
    "agent() is not loaded. Run your complete agent code first."
)

assert callable(agent), (
    "agent exists but is not callable."
)


function_sources = {}

for name, obj in list(globals().items()):

    if name.startswith("__"):
        continue

    if inspect.isfunction(obj) or inspect.isclass(obj):

        try:
            src = inspect.getsource(obj)

            if src.strip():
                function_sources[name] = src

        except (
            OSError,
            IOError,
            TypeError
        ):
            pass


print("Functions/classes found:", len(function_sources))


def get_names(source):

    try:
        tree = ast.parse(
            source
        )
    except Exception:
        return set()

    return {
        node.id
        for node in ast.walk(tree)
        if isinstance(node, ast.Name)
    }


needed = set()
queue = ["agent"]

while queue:

    name = queue.pop(0)

    if name in needed:
        continue

    if name not in function_sources:
        continue

    needed.add(name)

    for dependency in get_names(
        function_sources[name]
    ):

        if (
            dependency in function_sources
            and dependency not in needed
        ):

            queue.append(
                dependency
            )


ordered = []
visited = set()


def add_function(name):

    if name in visited:
        return

    if name not in needed:
        return

    visited.add(name)

    for dependency in get_names(
        function_sources[name]
    ):

        if dependency in needed:
            add_function(
                dependency
            )

    ordered.append(name)


add_function("agent")


imports = set()

for name in needed:

    for dependency in get_names(
        function_sources[name]
    ):

        if dependency not in globals():
            continue

        obj = globals()[dependency]

        if isinstance(
            obj,
            types.ModuleType
        ):

            module_name = obj.__name__

            if module_name not in {
                "__main__"
            }:

                imports.add(
                    f"import {module_name}"
                )


used_names = set()

for name in needed:

    used_names.update(
        get_names(
            function_sources[name]
        )
    )


constant_lines = []

for name in sorted(used_names):

    if name in needed:
        continue

    if name not in globals():
        continue

    if name.startswith("_"):
        continue

    value = globals()[name]

    if isinstance(
        value,
        (
            types.ModuleType,
            types.FunctionType,
            type
        )
    ):
        continue

    try:

        literal = repr(value)
        ast.literal_eval(
            literal
        )

        constant_lines.append(
            f"{name} = {literal}"
        )

    except Exception:
        pass


parts = []

parts.append(
    "# ============================================================\n"
    "# KAGGRICULTURE SUBMISSION AGENT\n"
    "# ============================================================\n"
)

# Imports
if imports:

    parts.append(
        "\n".join(
            sorted(imports)
        )
    )

# Constants
if constant_lines:

    parts.append(
        "\n".join(
            constant_lines
        )
    )

# Functions/classes
for name in ordered:

    parts.append(
        function_sources[name]
    )


main_code = "\n\n\n".join(
    parts
)


assert "def agent(" in main_code, (
    "Generated main.py does not contain agent()."
)


compile(
    main_code,
    str(OUTPUT),
    "exec"
)


OUTPUT.write_text(
    main_code,
    encoding="utf-8"
)


assert OUTPUT.exists()
assert OUTPUT.stat().st_size > 0


print()
print("=" * 80)
print("main.py CREATED")
print("=" * 80)

print("Path:", OUTPUT)
print("Size:", OUTPUT.stat().st_size, "bytes")

print()
print("Included functions/classes:")

for name in ordered:
    print("  ✓", name)


print()
print("=" * 80)
print("FRESH IMPORT TEST")
print("=" * 80)

fresh_namespace = {
    "__name__": "__submission__",
    "__file__": str(OUTPUT)
}

exec(
    compile(
        main_code,
        str(OUTPUT),
        "exec"
    ),
    fresh_namespace
)

assert "agent" in fresh_namespace
assert callable(
    fresh_namespace["agent"]
)

print("✓ main.py executes")
print("✓ agent() exists in fresh namespace")
print("✓ agent() is callable")

print()
print("=" * 80)
print("FINAL RESULT")
print("=" * 80)
print("✓ /kaggle/working/main.py")
print("✓ READY FOR KAGGLE SUBMISSION")
print()
print("IMPORTANT: SUBMIT main.py — NOT submission.py")

In [ ]:
from pathlib import Path
import shutil

main = Path("/kaggle/working/main.py")
submission = Path("/kaggle/working/submission.py")

print("=" * 70)
print("CREATING KAGGLE SUBMISSION FILE")
print("=" * 70)

assert main.exists(), "main.py does not exist"

# Copy the COMPLETE main.py
shutil.copy2(
    main,
    submission
)

print("main.py size:", main.stat().st_size, "bytes")
print("submission.py size:", submission.stat().st_size, "bytes")

assert submission.exists()
assert submission.stat().st_size == main.stat().st_size

text = submission.read_text(
    encoding="utf-8"
)

assert "def agent(" in text

# Syntax check
compile(
    text,
    str(submission),
    "exec"
)

print()
print("✓ submission.py CREATED")
print("✓ Same size as main.py")
print("✓ agent() found")
print("✓ Syntax check passed")
print()
print("READY TO SUBMIT")

In [ ]:
# ============================================================
# KAGGRICULTURE — FINAL SUBMISSION BUILDER
# Creates /kaggle/working/submission.py
# ============================================================

from pathlib import Path
import inspect

OUT = Path("/kaggle/working/submission.py")

print("=" * 80)
print("BUILDING FINAL KAGGRICULTURE SUBMISSION")
print("=" * 80)

# ------------------------------------------------------------
# 1. REQUIRED CONSTANTS
# ------------------------------------------------------------

constants_code = r'''
CROPS = {
    "WHEAT": {
        "seed": 10,
        "first_yield_day": 2,
        "max_yield_day": 4,
        "max_yield": 6,
        "ongoing": False
    },
    "CARROT": {
        "seed": 20,
        "first_yield_day": 2,
        "max_yield_day": 3,
        "max_yield": 4,
        "ongoing": False
    },
    "TOMATO": {
        "seed": 50,
        "first_yield_day": 8,
        "max_yield_day": 8,
        "max_yield": 4,
        "ongoing": True
    },
    "STRAWBERRY": {
        "seed": 100,
        "first_yield_day": 10,
        "max_yield_day": 10,
        "max_yield": 4,
        "ongoing": True
    },
    "MELON": {
        "seed": 80,
        "first_yield_day": 10,
        "max_yield_day": 12,
        "max_yield": 6,
        "ongoing": False
    }
}

LAND_ORDER = ["NE", "SW", "SE"]

LAND_PRICES = {
    "NE": 1000,
    "SW": 2000,
    "SE": 4000
}

FARM_HAND_COST_MULT = 10

BOARD_SIZE = 10
SHED_CAPACITY = 100

QUADRANT_TILES = {
    "NW": [
        (x, y)
        for y in range(0, 5)
        for x in range(0, 5)
    ],
    "NE": [
        (x, y)
        for y in range(0, 5)
        for x in range(5, 10)
    ],
    "SW": [
        (x, y)
        for y in range(5, 10)
        for x in range(0, 5)
    ],
    "SE": [
        (x, y)
        for y in range(5, 10)
        for x in range(5, 10)
    ]
}
'''

# ------------------------------------------------------------
# 2. FUNCTIONS REQUIRED BY agent()
# ------------------------------------------------------------

required = [
    "get_private",
    "get_seeds",
    "distance",
    "get_day",
    "normalize_position",
    "extract_hand_position",
    "get_unit_positions",
    "get_player_farm",
    "is_weed",
    "is_plant",
    "needs_water",
    "is_empty",
    "is_mature",
    "unlocked_tiles",
    "tile_at",
    "build_tasks",
    "get_shed",
    "move_toward",
    "unit_actions",
    "hire_cost",
    "get_market_prices",
    "economy_orders",
    "agent",
]

# ------------------------------------------------------------
# 3. COLLECT FUNCTIONS FROM CURRENT NOTEBOOK
# ------------------------------------------------------------

parts = [constants_code]
missing = []

for name in required:
    obj = globals().get(name)

    if obj is None or not callable(obj):
        missing.append(name)
        continue

    try:
        src = inspect.getsource(obj)
        parts.append("\n\n" + src)
        print("✓", name)
    except Exception as e:
        missing.append(name)
        print("✗", name, "->", repr(e))

if missing:
    raise RuntimeError(
        "These required functions are missing or could not be extracted:\n"
        + "\n".join(" - " + x for x in missing)
        + "\n\nDo NOT submit yet. Run the cells defining these functions first."
    )

# ------------------------------------------------------------
# 4. WRITE submission.py
# ------------------------------------------------------------

submission_code = "\n".join(parts).strip() + "\n"

OUT.write_text(
    submission_code,
    encoding="utf-8"
)

print()
print("=" * 80)
print("SUBMISSION FILE CREATED")
print("=" * 80)

print("Path:", OUT)
print("Size:", OUT.stat().st_size, "bytes")

# ------------------------------------------------------------
# 5. SYNTAX CHECK
# ------------------------------------------------------------

compile(
    submission_code,
    str(OUT),
    "exec"
)

print("✓ Python syntax valid")

# ------------------------------------------------------------
# 6. FRESH IMPORT TEST
# ------------------------------------------------------------

fresh = {}

exec(
    compile(
        submission_code,
        str(OUT),
        "exec"
    ),
    fresh
)

assert "agent" in fresh
assert callable(fresh["agent"])

print("✓ Fresh execution successful")
print("✓ agent() exists")
print("✓ agent() is callable")

# ------------------------------------------------------------
# 7. FINAL FILE CHECK
# ------------------------------------------------------------

assert OUT.exists()
assert OUT.stat().st_size > 0
assert "def agent(" in submission_code

print()
print("=" * 80)
print("FINAL RESULT")
print("=" * 80)

print("submission.py exists:", OUT.exists())
print("submission.py size:", OUT.stat().st_size, "bytes")
print("contains agent():", "def agent(" in submission_code)

print()
print("READY FOR KAGGLE")
print("=" * 80)